# TurbulenceDiffusion — independent magnetic k-ε vs molecular diffusion

`MagneticStandalone` · comparison of the new magnetic k-ε turbulence
modelisation against the legacy molecular diffusion resolution (TRUST-LiMe).


## Introduction

Both cases solve the standalone induction equation on the same PolyMAC mesh,
same boundary/initial data (`sin(pi*y)`), same η = 0.05, same time window.
The only difference is the turbulence model:

- **molecular** (legacy): no model, constant diffusivity.
- **k-ε** (new): `modele_turbulence_magnetique`, which adds
  η_t = C_μ_b k_b²/(ε_b Pm_t_b) on top of η (k_b = 0.5, ε_b = 1.0).

Expected: the k-ε case decays faster (larger effective diffusivity).


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers


In [ ]:
import numpy as np
import os
import pathlib
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    matches = list(pathlib.Path(BUILD).rglob(f"{case}_{name}.son"))
    assert matches, f"no probe file {case}_{name}.son under {BUILD}"
    path = str(matches[0])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def rate(case, tags):
    vals = []
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0 or t[-1] == 0:
            continue
        vals.append(-np.log(abs(b[-1] / b[0])) / t[-1])
    return float(np.mean(vals)) if vals else float("nan")

TAGS = ["SONDE_Y050"]
ETA = 0.05
LAM_MOL = ETA * np.pi**2
LAM_KE  = (ETA + 0.09 * 0.5**2 / 1.0) * np.pi**2
print("BUILD =", BUILD)


## Run


In [ ]:
from trustutils import run

NB_PROCS = 1

run.reset()
cases = {}
cases["kepsilon"] = run.addCaseFromTemplate("turbulence_diffusion_kepsilon.data",
                                           "kepsilon", dic={"nx": 17, "ny": 65},
                                           nbProcs=NB_PROCS, excluNR=True)
cases["molecular"] = run.addCaseFromTemplate("turbulence_diffusion_molecular.data",
                                            "molecular", dic={"nx": 17, "ny": 65},
                                            nbProcs=NB_PROCS, excluNR=True)
run.printCases()
run.runCases()
print("Cases finished.")


## Checks


In [ ]:
lam_mol = rate(cases["molecular"].dataFileName_, TAGS)
lam_ke  = rate(cases["kepsilon"].dataFileName_, TAGS)
print(f"molecular : lambda = {lam_mol:.6f} (expected {LAM_MOL:.6f})")
print(f"k-epsilon : lambda = {lam_ke:.6f} (expected ~{LAM_KE:.6f})")
print(f"ratio k-eps / molecular = {lam_ke/lam_mol:.3f}")

rel_mol = abs(lam_mol - LAM_MOL) / LAM_MOL
print(f"molecular relative error = {rel_mol:.3e}")
assert rel_mol < 0.1, f"molecular decay off: rel. err {rel_mol:.3e}"
print("OK: molecular case matches exp(-eta pi^2 t).")

assert lam_ke > lam_mol, f"k-epsilon not faster: {lam_ke} vs {lam_mol}"
print("OK: k-epsilon decays faster than molecular.")


In [ ]:
import matplotlib.pyplot as plt

t_m, b_m = read_son(cases["molecular"].dataFileName_, "SONDE_Y050")
t_k, b_k = read_son(cases["kepsilon"].dataFileName_, "SONDE_Y050")
plt.figure(figsize=(6.5, 4.0))
plt.plot(t_m, b_m / b_m[0], "o-", ms=2, label="molecular")
plt.plot(t_k, b_k / b_k[0], "s-", ms=2, label="k-epsilon")
plt.plot(t_m, np.exp(-LAM_MOL * np.asarray(t_m)), "k--", label="analytic molecular")
plt.xlabel("t [s]"); plt.ylabel("b(t)/b(0)"); plt.legend(); plt.grid(True); plt.show()


## Result


In [ ]:
print("TurbulenceDiffusion: PASS (independent magnetic k-epsilon vs molecular).")


## Computer Performance


In [ ]:
run.tablePerf()
